In [1]:
from pathlib import Path
import pandas as pd

file_path = Path("../Dataset_File/Online Retail.xlsx")

if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {file_path.resolve()}"
    )

df_raw = pd.read_excel(file_path)

df_prep = df_raw.copy()

print("Original Dataset Shape:", df_raw.shape)
print("Working Dataset Shape:", df_prep.shape)

print("\nDataset Columns:")
print(df_prep.columns.tolist())

Original Dataset Shape: (541909, 8)
Working Dataset Shape: (541909, 8)

Dataset Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


# Data Preparation

## 1. Creating a Working Copy

در اولین مرحله از آماده‌سازی داده، یک نسخه مستقل از دیتاست اصلی ایجاد می‌کنیم.

هدف این کار حفظ داده‌های اولیه و جلوگیری از تغییر ناخواسته آن‌ها در طول فرآیند آماده‌سازی است.

در این مرحله دو دیتافریم ایجاد می‌شود:

- `df_raw`: نسخه اصلی دیتاست که بدون تغییر نگهداری می‌شود.
- `df_prep`: نسخه کاری که تمام عملیات Data Preparation روی آن انجام خواهد شد.

با این روش، در صورت بروز خطا یا نیاز به بازگشت به داده‌های اولیه، نسخه اصلی همچنان در دسترس خواهد بود.

In [6]:

text_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Country"
]

for column in text_columns:
    df_prep[column] = df_prep[column].astype("string")



customer_id_numeric = pd.to_numeric(
    df_prep["CustomerID"],
    errors="raise"
)

non_integer_ids = (
    customer_id_numeric.dropna() % 1 != 0
).sum()

if non_integer_ids > 0:
    raise ValueError("CustomerID contains non-integer values.")

df_prep["CustomerID"] = customer_id_numeric.astype("Int64")



df_prep["InvoiceDate"] = pd.to_datetime(
    df_prep["InvoiceDate"],
    errors="raise"
)

df_prep["Quantity"] = pd.to_numeric(
    df_prep["Quantity"],
    errors="raise"
)

df_prep["UnitPrice"] = pd.to_numeric(
    df_prep["UnitPrice"],
    errors="raise"
)



print("Updated Data Types:")
print(df_prep.dtypes)

print("\nMissing CustomerID:", df_prep["CustomerID"].isna().sum())

print("Dataset Shape:", df_prep.shape)

Updated Data Types:
InvoiceNo              string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              Int64
Country                string
dtype: object

Missing CustomerID: 135080
Dataset Shape: (541909, 8)


## 2. Data Type Correction

در این مرحله نوع داده‌های ستون‌های دیتاست را بر اساس ماهیت واقعی آن‌ها اصلاح می‌کنیم.

شناسه‌هایی مانند `InvoiceNo` و `StockCode` به نوع String تبدیل می‌شوند؛ زیرا این مقادیر شناسه هستند و معنای عددی ندارند.

همچنین ستون `CustomerID` که به دلیل وجود مقادیر گمشده از نوع Float بود، به Nullable Integer تبدیل می‌شود.

برای ستون‌های `InvoiceDate`، `Quantity` و `UnitPrice` نیز نوع داده مناسب بررسی و تثبیت می‌شود.

هدف این مرحله، یکسان‌سازی نوع داده‌ها و آماده‌سازی آن‌ها برای عملیات بعدی مانند فیلترکردن تراکنش‌ها، محاسبات زمانی و استخراج ویژگی‌های مشتری است.

در این مرحله هیچ رکوردی حذف نمی‌شود و مقادیر گمشده نیز دست‌نخورده باقی می‌مانند.

In [7]:

duplicate_mask = df_prep.duplicated(keep="first")

duplicate_count = duplicate_mask.sum()

duplicate_percent = (
    duplicate_count / len(df_prep)
) * 100


print("Duplicate rows:", duplicate_count)

print(
    "Duplicate percentage:",
    round(duplicate_percent, 2),
    "%"
)

print("\nExamples of duplicate records:")

display(
    df_prep[df_prep.duplicated(keep=False)].head(10)
)

Duplicate rows: 5268
Duplicate percentage: 0.97 %

Examples of duplicate records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920,United Kingdom


## 3. Duplicate Records Identification

در مرحله Data Understanding مشخص شد که دیتاست شامل 5,268 رکورد تکراری کامل است.

در این مرحله مجدداً با استفاده از تابع `duplicated()` وجود رکوردهای تکراری را در نسخه آماده‌سازی‌شده بررسی می‌کنیم.

منظور از رکورد تکراری کامل، ردیفی است که تمام مقادیر آن در هر هشت ستون با یک ردیف دیگر یکسان باشد.

وجود این رکوردها می‌تواند باعث شمارش چندباره تراکنش‌ها و محاسبه نادرست ویژگی‌های رفتاری مشتریان، به‌خصوص Frequency و Monetary شود.

پس از بررسی نمونه‌ها، نسخه‌های اضافی رکوردهای کاملاً یکسان حذف خواهند شد و اولین نسخه هر رکورد حفظ می‌شود.

In [8]:

rows_before = len(df_prep)

duplicates_before = df_prep.duplicated().sum()


df_prep = (
    df_prep
    .drop_duplicates(keep="first")
    .reset_index(drop=True)
)


rows_after = len(df_prep)

duplicates_after = df_prep.duplicated().sum()

removed_rows = rows_before - rows_after


print("Rows before:", rows_before)
print("Duplicate rows found:", duplicates_before)

print("\nRows removed:", removed_rows)
print("Rows after:", rows_after)

print("\nRemaining duplicates:", duplicates_after)
print("Final Dataset Shape:", df_prep.shape)

Rows before: 541909
Duplicate rows found: 5268

Rows removed: 5268
Rows after: 536641

Remaining duplicates: 0
Final Dataset Shape: (536641, 8)


## 3. Duplicate Records Removal

پس از شناسایی رکوردهای تکراری، تصمیم به حذف نسخه‌های اضافی رکوردهای کاملاً یکسان گرفتیم.

برای این کار از تابع `drop_duplicates()` استفاده شد.

با تنظیم پارامتر `keep="first"`، اولین نسخه از هر رکورد حفظ و نسخه‌های تکراری بعدی حذف شدند.

سپس با استفاده از `reset_index()` شماره ردیف‌های دیتاست مجدداً تنظیم شد.

در پایان تعداد رکوردهای قبل و بعد از حذف مقایسه شد تا از صحت عملیات اطمینان حاصل شود.

هدف این مرحله جلوگیری از شمارش چندباره تراکنش‌ها و کاهش خطا در محاسبه ویژگی‌های رفتاری مشتریان، به‌خصوص Frequency و Monetary است.

In [9]:

missing_count = df_prep.isna().sum()

missing_percent = df_prep.isna().mean() * 100



missing_report = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percent": missing_percent
})



display(
    missing_report.sort_values(
        by="Missing_Percent",
        ascending=False
    )
)

,Missing_Count,Missing_Percent
CustomerID,135037,25.163377
Description,1454,0.270945
StockCode,0,0.000000
InvoiceNo,0,0.000000
Quantity,0,0.000000
InvoiceDate,0,0.000000
UnitPrice,0,0.000000
Country,0,0.000000


## 4. Missing Values Handling

در مرحله Data Understanding مشخص شد که دو ستون CustomerID و Description دارای مقادیر گمشده هستند.

ستون CustomerID دارای بیشترین میزان داده گمشده است و به دلیل ماهیت پروژه، اهمیت زیادی دارد.

از آنجایی که هدف پروژه تحلیل رفتار خرید مشتریان است، تراکنش‌های فاقد CustomerID را نمی‌توان به مشتری مشخصی نسبت داد.

بنابراین تصمیم گرفته شد این رکوردها برای تحلیل‌های مشتری‌محور مورد استفاده قرار نگیرند؛ اما فعلاً از دیتاست اصلی حذف نمی‌شوند و به‌صورت جداگانه نگهداری خواهند شد.

در ستون Description نیز به دلیل اینکه شناسه محصول (StockCode) موجود است و توضیحات محصول برای محاسبات اصلی RFM ضروری نیست، مقادیر گمشده بدون جایگزینی حفظ می‌شوند.

در این مرحله ابتدا تعداد و درصد مقادیر گمشده مجدداً بررسی می‌شود تا تصمیمات آماده‌سازی بر اساس وضعیت فعلی دیتاست انجام شوند.

In [10]:

missing_customer_mask = df_prep["CustomerID"].isna()

df_no_customer_id = df_prep.loc[
    missing_customer_mask
].copy()

df_customer = df_prep.loc[
    ~missing_customer_mask
].copy()



print("Original Dataset:", df_prep.shape)

print("\nCustomer Identified Records:", df_customer.shape)

print("Records without CustomerID:", df_no_customer_id.shape)



missing_customer_percentage = (
    len(df_no_customer_id) / len(df_prep)
) * 100

print(
    "\nMissing CustomerID Percentage:",
    round(missing_customer_percentage, 2),
    "%"
)



print(
    "\nMissing Description in Customer Dataset:",
    df_customer["Description"].isna().sum()
)

Original Dataset: (536641, 8)

Customer Identified Records: (401604, 8)
Records without CustomerID: (135037, 8)

Missing CustomerID Percentage: 25.16 %

Missing Description in Customer Dataset: 0


## 4.2. Handling Missing CustomerID

در این مرحله رکوردها را بر اساس وجود یا عدم وجود شناسه مشتری تفکیک می‌کنیم.

از آنجایی که هدف پروژه تحلیل رفتار مشتریان و پیش‌بینی ریزش آن‌هاست، وجود CustomerID برای نسبت‌دادن تراکنش‌ها به مشتریان مشخص ضروری است.

بنابراین دو دیتافریم ایجاد می‌کنیم:

- `df_customer`: شامل تراکنش‌هایی است که دارای شناسه مشتری هستند.
- `df_no_customer_id`: شامل تراکنش‌هایی است که فاقد شناسه مشتری هستند.

رکوردهای فاقد CustomerID برای تحلیل‌های مشتری‌محور و ساخت ویژگی‌های RFM قابل استفاده نیستند؛ زیرا نمی‌توان آن‌ها را به مشتری مشخصی نسبت داد.

با این حال، این رکوردها از دیتاست اصلی حذف نمی‌شوند و به‌صورت جداگانه نگهداری می‌شوند.

در پایان، تعداد و درصد رکوردهای هر گروه بررسی می‌شود.

In [15]:

cancelled_mask = (
    df_prep["InvoiceNo"].str.startswith("C", na=False)
)



df_cancelled_all = df_prep.loc[
    cancelled_mask
].copy()



df_customer_cancellations = df_customer.loc[
    df_customer["InvoiceNo"].str.startswith("C", na=False)
].copy()



df_unassigned_cancellations = df_no_customer_id.loc[
    df_no_customer_id["InvoiceNo"].str.startswith("C", na=False)
].copy()



df_customer_non_cancelled = df_customer.loc[
    ~df_customer["InvoiceNo"].str.startswith("C", na=False)
].copy()

In [16]:
print("Total cancelled records:", len(df_cancelled_all))

print(
    "Cancelled records with CustomerID:",
    len(df_customer_cancellations)
)

print(
    "Cancelled records without CustomerID:",
    len(df_unassigned_cancellations)
)

print(
    "\nNon-cancelled customer records:",
    len(df_customer_non_cancelled)
)



print(
    "\nUnique cancelled invoices:",
    df_cancelled_all["InvoiceNo"].nunique()
)



print(
    "Customers with cancellations:",
    df_customer_cancellations["CustomerID"].nunique()
)



assert (
    len(df_customer_cancellations)
    + len(df_unassigned_cancellations)
    == len(df_cancelled_all)
)

assert (
    len(df_customer_cancellations)
    + len(df_customer_non_cancelled)
    == len(df_customer)
)

print("\nSeparation checks passed successfully.")

Total cancelled records: 9251
Cancelled records with CustomerID: 8872
Cancelled records without CustomerID: 379

Non-cancelled customer records: 392732

Unique cancelled invoices: 3836
Customers with cancellations: 1589

Separation checks passed successfully.


## 5. Cancellation Handling

در مرحله Data Understanding مشخص شد که فاکتورهایی که شماره آن‌ها با حرف C شروع می‌شود، نشان‌دهنده فاکتورهای لغوشده هستند.

همچنین بررسی مقادیر Quantity نشان داد که بخش عمده تراکنش‌های دارای Quantity منفی مربوط به فاکتورهای لغوشده است.

از آنجایی که لغو و مرجوعی می‌تواند بخشی از رفتار واقعی مشتری باشد، تصمیم گرفتیم این تراکنش‌ها را بدون بررسی حذف نکنیم.

در این مرحله تراکنش‌ها به گروه‌های زیر تفکیک می‌شوند:

- `df_customer_cancellations`: فاکتورهای لغوشده دارای CustomerID.
- `df_unassigned_cancellations`: فاکتورهای لغوشده فاقد CustomerID.
- `df_customer_non_cancelled`: تراکنش‌های غیرلغوشده مشتریان.

تراکنش‌های لغوشده دارای CustomerID می‌توانند در مراحل بعدی برای استخراج ویژگی‌های مرتبط با رفتار لغو و مرجوعی استفاده شوند.

همچنین تراکنش‌های فاقد شناسه مشتری به‌صورت جداگانه نگهداری می‌شوند؛ زیرا امکان انتساب آن‌ها به مشتری مشخص وجود ندارد.

در این مرحله هیچ‌کدام از این گروه‌ها به‌صورت مخرب از داده‌های اصلی حذف نمی‌شوند.

In [17]:

cancelled_mask = (
    df_prep["InvoiceNo"].str.startswith("C", na=False)
)


negative_quantity_mask = df_prep["Quantity"] < 0


df_negative_cancellations = df_prep.loc[
    negative_quantity_mask & cancelled_mask
].copy()


df_internal_adjustments = df_prep.loc[
    negative_quantity_mask & ~cancelled_mask
].copy()


df_bad_debt_adjustments = df_prep.loc[
    df_prep["UnitPrice"] < 0
].copy()

In [18]:

groups = {
    "Negative Cancellations": df_negative_cancellations,
    "Internal Adjustments": df_internal_adjustments,
    "Bad Debt Adjustments": df_bad_debt_adjustments
}

for name, data in groups.items():

    print(f"\n{name}")
    print("-" * 40)

    print("Number of rows:", len(data))

    print(
        "Missing CustomerID:",
        data["CustomerID"].isna().sum()
    )

    print(
        "Zero UnitPrice:",
        (data["UnitPrice"] == 0).sum()
    )

    print(
        "Missing Description:",
        data["Description"].isna().sum()
    )


Negative Cancellations
----------------------------------------
Number of rows: 9251
Missing CustomerID: 379
Zero UnitPrice: 0
Missing Description: 0

Internal Adjustments
----------------------------------------
Number of rows: 1336
Missing CustomerID: 1336
Zero UnitPrice: 1336
Missing Description: 862

Bad Debt Adjustments
----------------------------------------
Number of rows: 2
Missing CustomerID: 2
Zero UnitPrice: 0
Missing Description: 0


In [19]:
print("Internal Adjustment Descriptions:")

display(
    df_internal_adjustments["Description"]
    .value_counts(dropna=False)
    .head(15)
)


print("\nBad Debt Adjustment Records:")

display(
    df_bad_debt_adjustments[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "CustomerID"
        ]
    ]
)

Internal Adjustment Descriptions:


Description
<NA>                      862
check                     120
damages                    45
damaged                    42
?                          41
sold as set on dotcom      20
Damaged                    14
thrown away                 9
Unsaleable, destroyed.      9
??                          7
damages?                    5
wet damaged                 5
ebay                        5
smashed                     4
missing                     3
Name: count, dtype: Int64


Bad Debt Adjustment Records:


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
297646,A563186,B,Adjust bad debt,1,-11062.06,<NA>
297647,A563187,B,Adjust bad debt,1,-11062.06,<NA>


## 6. Negative Quantity and UnitPrice Handling

در مرحله Data Understanding مشخص شد که مقادیر منفی در ستون‌های Quantity و UnitPrice الزاماً نشان‌دهنده خطای داده نیستند.

بررسی‌ها سه گروه متفاوت را مشخص کردند:

1. Cancellation Records:
   ردیف‌هایی با Quantity منفی و شماره فاکتور شروع‌شونده با C.
   این رکوردها می‌توانند نشان‌دهنده لغو یا مرجوعی کالا توسط مشتری باشند و در صورت وجود CustomerID برای تحلیل رفتار مشتری قابل استفاده‌اند.

2. Internal Adjustments:
   ردیف‌هایی با Quantity منفی که شماره فاکتور آن‌ها با C شروع نمی‌شود.
   بررسی اولیه نشان داد این رکوردها فاقد CustomerID و دارای UnitPrice صفر هستند و توضیحات آن‌ها شامل مواردی مانند damages، check و thrown away است.

3. Bad Debt Adjustments:
   دو رکورد دارای UnitPrice منفی با توضیحات Adjust bad debt و StockCode برابر B.
   این رکوردها تعدیلات مالی هستند و خرید واقعی مشتری محسوب نمی‌شوند.

بنابراین، به‌جای حذف تمام مقادیر منفی، رکوردها بر اساس ماهیت تجاری آن‌ها تفکیک شدند.

تراکنش‌های مرتبط با لغو و مرجوعی مشتری حفظ می‌شوند تا در صورت نیاز برای استخراج ویژگی‌های رفتاری استفاده شوند.

تعدیلات داخلی و مالی نیز از محاسبات مشتری‌محور کنار گذاشته خواهند شد.

در این مرحله هیچ رکوردی از دیتاست اصلی حذف نشده است.

In [20]:

non_product_codes = [
    "POST",
    "DOT",
    "M",
    "m",
    "C2",
    "D",
    "S",
    "AMAZONFEE",
    "CRUK",
    "B",
    "BANK CHARGES",
    "PADS"
]



non_product_mask = df_prep["StockCode"].isin(
    non_product_codes
)


df_non_product = df_prep.loc[
    non_product_mask
].copy()


print("Non-product records:", len(df_non_product))

Non-product records: 2910


## تعریف کدهای غیرکالایی

ابتدا کدهایی را که در Data Understanding شناسایی کردیم مشخص می‌کنیم

In [21]:

df_non_product_analysis = df_non_product.assign(
    LineTotal=(
        df_non_product["Quantity"]
        * df_non_product["UnitPrice"]
    )
)



non_product_report = (
    df_non_product_analysis
    .groupby("StockCode")
    .agg(
        Rows=("InvoiceNo", "size"),

        Description=("Description", "first"),

        Rows_with_Customer=(
            "CustomerID",
            lambda x: x.notna().sum()
        ),

        Net_Value=("LineTotal", "sum")
    )
    .sort_values("Rows", ascending=False)
)


display(non_product_report.round(2))

,Rows,Description,Rows_with_Customer,Net_Value
StockCode,,,,
POST,1256,POSTAGE,1196,66230.64
DOT,710,DOTCOM POSTAGE,16,206245.48
M,566,Manual,460,-69034.19
C2,144,CARRIAGE,134,6986.00
D,77,Discount,77,-5696.22
S,62,SAMPLES,0,-3039.65
BANK CHARGES,37,Bank Charges,12,-7175.64
AMAZONFEE,34,AMAZON FEE,0,-221520.50
CRUK,16,CRUK Commission,16,-7933.43


## بررسی تراکنش‌های غیرکالایی

حالا می‌خواهیم بدانیم از هر کد چه تعداد رکورد داریم و ارزش مالی آن‌ها چقدر است

In [22]:

gift_voucher_mask = (
    df_prep["StockCode"].str.startswith(
        "gift_",
        na=False
    )
)


df_gift_vouchers = df_prep.loc[
    gift_voucher_mask
].copy()



gift_voucher_report = (
    df_gift_vouchers["StockCode"]
    .value_counts()
    .rename_axis("StockCode")
    .reset_index(name="Rows")
)


display(gift_voucher_report)


print("Total gift voucher records:", len(df_gift_vouchers))

,StockCode,Rows
0,gift_0001_20,10
1,gift_0001_10,9
2,gift_0001_30,8
3,gift_0001_50,4
4,gift_0001_40,3


Total gift voucher records: 34


## بررسی کارت‌های هدیه

در Data Understanding متوجه شدیم که کدهایی با الگوی gift_0001_* نیز در دیتاست وجود دارند.

این کدها مربوط به کارت هدیه هستند.

نکته مهم این است که کارت هدیه نه یک محصول معمولی است و نه لزوماً یک هزینه جانبی مانند پست.

بنابراین آن‌ها را جداگانه نگهداری می‌کنیم

In [23]:

remaining_mask = (
    ~non_product_mask
    & ~gift_voucher_mask
)


df_other_stock_codes = df_prep.loc[
    remaining_mask
].copy()


print(
    "Other StockCode records:",
    len(df_other_stock_codes)
)


print(
    "Unique remaining StockCodes:",
    df_other_stock_codes["StockCode"].nunique()
)


print("\nSample records:")

display(
    df_other_stock_codes[
        ["StockCode", "Description", "Quantity", "UnitPrice"]
    ].head(10)
)

Other StockCode records: 533697
Unique remaining StockCodes: 4053

Sample records:


,StockCode,Description,Quantity,UnitPrice
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55
1,71053,WHITE METAL LANTERN,6,3.39
2,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,3.39
5,22752,SET 7 BABUSHKA NESTING BOXES,2,7.65
6,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,4.25
7,22633,HAND WARMER UNION JACK,6,1.85
8,22632,HAND WARMER RED POLKA DOT,6,1.85
9,84879,ASSORTED COLOUR BIRD ORNAMENT,32,1.69


## بررسی کدهایی که هنوز طبقه‌بندی نشده‌اند

حالا سایر رکوردها را جدا می‌کنیم.

دقت کن که در این مرحله به آن‌ها برچسب قطعی محصول نمی‌دهیم؛ فقط می‌دانیم در فهرست کدهای غیرکالایی شناخته‌شده و کارت‌های هدیه قرار ندارند

## 7. Non-Product Transactions Identification

در مرحله Data Understanding مشخص شد که همه مقادیر ستون StockCode نماینده محصولات واقعی نیستند.

برخی از کدها مربوط به هزینه‌های جانبی، تعدیلات مالی، تخفیف‌ها و ثبت‌های دستی هستند.

کدهای شناسایی‌شده شامل POST، DOT، M، C2، D، S، BANK CHARGES، AMAZONFEE، CRUK، PADS، B و m هستند.

وجود این رکوردها می‌تواند باعث ایجاد خطا در محاسبه ویژگی‌های رفتاری مشتری شود.

برای مثال، هزینه پست نباید به‌عنوان یک محصول مستقل در محاسبه UniqueProducts در نظر گرفته شود.

در این مرحله کدهای غیرکالایی شناسایی و در یک دیتافریم جداگانه نگهداری می‌شوند.

همچنین کدهای مربوط به کارت هدیه با الگوی gift_ به‌صورت مستقل شناسایی می‌شوند؛ زیرا ماهیت آن‌ها با سایر تراکنش‌های غیرکالایی متفاوت است.

نکته مهم این است که تمام تراکنش‌های غیرکالایی الزاماً فاقد ارزش مالی نیستند.

به همین دلیل، هزینه‌هایی مانند POSTAGE و CARRIAGE نباید بدون بررسی از Monetary حذف شوند.

در این مرحله هیچ رکوردی حذف نمی‌شود و هدف صرفاً شناسایی و دسته‌بندی تراکنش‌ها برای تصمیم‌گیری در مراحل بعدی است.

In [25]:

cancelled_mask = (
    df_prep["InvoiceNo"].str.startswith("C", na=False)
)

valid_customer_mask = (
    df_prep["CustomerID"].notna()
)

valid_quantity_mask = (
    df_prep["Quantity"] > 0
)

valid_price_mask = (
    df_prep["UnitPrice"] > 0
)

valid_date_mask = (
    df_prep["InvoiceDate"].notna()
)


valid_purchase_mask = (
    valid_customer_mask
    & ~cancelled_mask
    & valid_quantity_mask
    & valid_price_mask
    & valid_date_mask
)



df_purchase_candidates = df_prep.loc[
    valid_purchase_mask
].copy()


print("Valid purchase candidates:", len(df_purchase_candidates))

Valid purchase candidates: 392692


 ## شناسایی خریدهای معتبر اولیه

ابتدا شرایط اولیه خرید معتبر را اعمال می‌کنیم

In [26]:

non_product_codes = [
    "POST",
    "DOT",
    "M",
    "m",
    "C2",
    "D",
    "S",
    "AMAZONFEE",
    "CRUK",
    "B",
    "BANK CHARGES",
    "PADS"
]



gift_voucher_mask = (
    df_purchase_candidates["StockCode"]
    .str.startswith("gift_", na=False)
)



non_product_mask = (
    df_purchase_candidates["StockCode"]
    .isin(non_product_codes)
)



df_customer_gift_vouchers = df_purchase_candidates.loc[
    gift_voucher_mask
].copy()



df_customer_non_product = df_purchase_candidates.loc[
    non_product_mask
].copy()



df_valid_product_sales = df_purchase_candidates.loc[
    ~non_product_mask & ~gift_voucher_mask
].copy()

تفکیک خرید کالا از هزینه‌های جانبی

In [27]:
print("Purchase Candidates:", len(df_purchase_candidates))

print(
    "Valid Product Sales:",
    len(df_valid_product_sales)
)

print(
    "Non-Product Transactions:",
    len(df_customer_non_product)
)

print(
    "Gift Voucher Transactions:",
    len(df_customer_gift_vouchers)
)


print("\nUnique Customers:",
      df_valid_product_sales["CustomerID"].nunique())

print("Unique Invoices:",
      df_valid_product_sales["InvoiceNo"].nunique())

print("Unique Products:",
      df_valid_product_sales["StockCode"].nunique())



assert df_valid_product_sales["CustomerID"].notna().all()

assert (df_valid_product_sales["Quantity"] > 0).all()

assert (df_valid_product_sales["UnitPrice"] > 0).all()

assert not df_valid_product_sales["InvoiceNo"].str.startswith(
    "C", na=False
).any()

assert not df_valid_product_sales["StockCode"].isin(
    non_product_codes
).any()

assert not df_valid_product_sales["StockCode"].str.startswith(
    "gift_", na=False
).any()

print("\nAll validation checks passed!")

Purchase Candidates: 392692
Valid Product Sales: 391150
Non-Product Transactions: 1542
Gift Voucher Transactions: 0

Unique Customers: 4334
Unique Invoices: 18402
Unique Products: 3659

All validation checks passed!


بررسی صحت دیتاست نهایی

حالا باید مطمئن شویم فیلترها درست عمل کرده‌اند

## 8. Valid Purchase Transactions

در این مرحله با استفاده از یافته‌های Data Understanding و تصمیمات مراحل قبلی، دیتاست خریدهای معتبر مشتریان را ایجاد می‌کنیم.

ابتدا رکوردهایی انتخاب می‌شوند که دارای CustomerID هستند، فاکتور آن‌ها لغوشده نیست، Quantity و UnitPrice مثبت دارند و InvoiceDate آن‌ها معتبر است.

سپس تراکنش‌های مربوط به هزینه‌های جانبی و سایر کدهای غیرکالایی از محصولات واقعی تفکیک می‌شوند.

همچنین تراکنش‌های مربوط به کارت هدیه به‌صورت مستقل نگهداری می‌شوند.

در نتیجه سه گروه ایجاد می‌شود:

- `df_valid_product_sales`: خرید محصولات واقعی مشتریان.
- `df_customer_non_product`: تراکنش‌های غیرکالایی دارای CustomerID.
- `df_customer_gift_vouchers`: تراکنش‌های مربوط به کارت هدیه.

دیتافریم `df_valid_product_sales` مبنای اولیه استخراج ویژگی‌های رفتاری مشتریان خواهد بود.

هزینه‌های جانبی و کارت‌های هدیه حذف دائمی نمی‌شوند؛ زیرا ممکن است در محاسبات مالی مشتریان اهمیت داشته باشند.

در پایان، با استفاده از شروط اعتبارسنجی، صحت فیلترهای اعمال‌شده بررسی می‌شود.

In [28]:

history_days = 180
future_days = 90


valid_dates = df_prep["InvoiceDate"].dropna()

if valid_dates.empty:
    raise ValueError("No valid InvoiceDate values found.")


dataset_start = valid_dates.min().normalize()

dataset_end = valid_dates.max().normalize()


print("Dataset Start:", dataset_start)

print("Dataset End:", dataset_end)

Dataset Start: 2010-12-01 00:00:00
Dataset End: 2011-12-09 00:00:00


## تعیین محدوده زمانی دیتاست

ابتدا تاریخ شروع و پایان دیتاست را محاسبه می‌کنیم

In [29]:

earliest_reference_date = (
    dataset_start + pd.Timedelta(days=history_days)
)

latest_reference_date = (
    dataset_end - pd.Timedelta(days=future_days)
)


print("Earliest Reference Date:", earliest_reference_date)

print("Latest Reference Date:", latest_reference_date)



if earliest_reference_date > latest_reference_date:

    raise ValueError(
        "Insufficient dataset duration for the required time windows."
    )

else:

    print("\nA feasible reference date range exists.")

Earliest Reference Date: 2011-05-30 00:00:00
Latest Reference Date: 2011-09-10 00:00:00

A feasible reference date range exists.


## محاسبه محدوده تاریخ‌های مرجع ممکن

حالا می‌خواهیم ببینیم چه تاریخ‌هایی برای Reference Date قابل استفاده هستند.

In [30]:

reference_date = latest_reference_date


feature_start_date = (
    reference_date - pd.Timedelta(days=history_days)
)

feature_end_date = reference_date

label_start_date = reference_date

label_end_date = (
    reference_date + pd.Timedelta(days=future_days)
)


print("Reference Date:", reference_date)

print("\nFeature Window:")
print("Start:", feature_start_date)
print("End:", feature_end_date)

print("\nLabel Window:")
print("Start:", label_start_date)
print("End:", label_end_date)


assert feature_start_date >= dataset_start

assert label_end_date <= (
    dataset_end + pd.Timedelta(days=1)
)

assert feature_end_date == label_start_date

print("\nAll date boundaries are valid.")

Reference Date: 2011-09-10 00:00:00

Feature Window:
Start: 2011-03-14 00:00:00
End: 2011-09-10 00:00:00

Label Window:
Start: 2011-09-10 00:00:00
End: 2011-12-09 00:00:00

All date boundaries are valid.


انتخاب Reference Date نهایی

حالا از بین تاریخ‌های ممکن، آخرین تاریخ را انتخاب می‌کنیم

## 9. Reference Date Selection

در این مرحله تاریخ مرجع پروژه (Reference Date) را تعیین می‌کنیم.

تاریخ مرجع نقطه‌ای است که اطلاعات گذشته و آینده مشتری را از یکدیگر جدا می‌کند.

طبق اهداف پروژه، دو پنجره زمانی تعریف شده است:

* Feature Window: شامل ۱۸۰ روز قبل از تاریخ مرجع برای استخراج ویژگی‌های رفتاری مشتریان.
* Label Window: شامل ۹۰ روز بعد از تاریخ مرجع برای تعیین وضعیت ریزش مشتریان.

بر اساس بررسی Data Understanding، دیتاست بازه زمانی ۱ دسامبر ۲۰۱۰ تا ۹ دسامبر ۲۰۱۱ را پوشش می‌دهد.

با توجه به محدودیت زمانی دیتاست، محدوده تاریخ‌های مرجع قابل استفاده از ۳۰ مه تا ۱۰ سپتامبر ۲۰۱۱ است.

برای استفاده از جدیدترین اطلاعات ممکن، تاریخ ۱۰ سپتامبر ۲۰۱۱ به‌عنوان Reference Date انتخاب شد.

بنابراین:

* Feature Window از ۱۴ مارس ۲۰۱۱ تا قبل از ۱۰ سپتامبر ۲۰۱۱ است.
* Label Window از ۱۰ سپتامبر ۲۰۱۱ تا قبل از ۱۰ دسامبر ۲۰۱۱ است.

در این طراحی، اطلاعات قبل از تاریخ مرجع فقط برای استخراج ویژگی‌ها استفاده می‌شوند و اطلاعات بعد از تاریخ مرجع صرفاً برای ساخت برچسب ریزش به کار می‌روند.

این تفکیک زمانی از نشت اطلاعات آینده (Data Leakage) جلوگیری می‌کند.


In [31]:

feature_window_mask = (
    df_valid_product_sales["InvoiceDate"].ge(feature_start_date)
    &
    df_valid_product_sales["InvoiceDate"].lt(reference_date)
)


df_feature_transactions = df_valid_product_sales.loc[
    feature_window_mask
].copy()


print("Feature Window Transactions:", len(df_feature_transactions))

print(
    "Unique Customers:",
    df_feature_transactions["CustomerID"].nunique()
)

print(
    "Unique Invoices:",
    df_feature_transactions["InvoiceNo"].nunique()
)

print(
    "Unique Products:",
    df_feature_transactions["StockCode"].nunique()
)

Feature Window Transactions: 156907
Unique Customers: 2783
Unique Invoices: 7979
Unique Products: 3210


## استخراج تراکنش‌های بازه ۱۸۰ روزه

در این مرحله از دیتافریم df_valid_product_sales استفاده می‌کنیم

In [32]:

print("Feature Start Date:", df_feature_transactions["InvoiceDate"].min())

print("Feature End Date:", df_feature_transactions["InvoiceDate"].max())


assert (
    df_feature_transactions["InvoiceDate"] >= feature_start_date
).all()

assert (
    df_feature_transactions["InvoiceDate"] < reference_date
).all()

assert df_feature_transactions["CustomerID"].notna().all()

assert len(df_feature_transactions) > 0

print("\nFeature Window validation passed!")

Feature Start Date: 2011-03-14 08:05:00
Feature End Date: 2011-09-09 15:53:00

Feature Window validation passed!


## بررسی صحت بازه زمانی

بعد از استخراج داده‌ها باید مطمئن شویم هیچ تراکنش خارج از بازه وارد نشده است

In [33]:

feature_summary = pd.DataFrame({
    "Metric": [
        "Total Transactions",
        "Unique Customers",
        "Unique Invoices",
        "Unique Products",
        "Total Units Sold",
        "Total Product Revenue"
    ],

    "Value": [
        len(df_feature_transactions),

        df_feature_transactions["CustomerID"].nunique(),

        df_feature_transactions["InvoiceNo"].nunique(),

        df_feature_transactions["StockCode"].nunique(),

        df_feature_transactions["Quantity"].sum(),

        (
            df_feature_transactions["Quantity"]
            * df_feature_transactions["UnitPrice"]
        ).sum()
    ]
})


display(feature_summary)

,Metric,Value
0,Total Transactions,156907.0
1,Unique Customers,2783.0
2,Unique Invoices,7979.0
3,Unique Products,3210.0
4,Total Units Sold,2153396.0
5,Total Product Revenue,3580816.3


بررسی آماری Feature Dataset

حالا می‌خواهیم شناخت اولیه‌ای از مشتریان موجود در این بازه داشته باشیم.

## 10. Feature Window Extraction

در این مرحله تراکنش‌های مربوط به بازه ۱۸۰ روز قبل از Reference Date استخراج می‌شوند.

هدف این مرحله ایجاد مجموعه داده‌ای است که رفتار گذشته مشتریان را نشان دهد.

با توجه به Reference Date انتخاب‌شده (2011-09-10)، بازه ویژگی‌ها به شکل زیر تعریف شده است:

* Start Date: 2011-03-14
* End Date: 2011-09-10 (Exclusive)

در این مرحله فقط تراکنش‌هایی انتخاب می‌شوند که تاریخ آن‌ها بزرگ‌تر یا مساوی تاریخ شروع و کوچک‌تر از تاریخ مرجع باشد.

دیتافریم `df_feature_transactions` شامل خریدهای معتبر مشتریان در این بازه خواهد بود.

این مجموعه داده در مراحل بعدی برای استخراج ویژگی‌هایی مانند Recency، Frequency، Monetary و سایر ویژگی‌های رفتاری مشتریان استفاده می‌شود.

همچنین با بررسی تعداد مشتریان، فاکتورها و محصولات منحصربه‌فرد، وضعیت داده‌های استخراج‌شده ارزیابی می‌شود.

**نکته مهم:** هیچ اطلاعاتی از تاریخ مرجع یا بعد از آن وارد Feature Dataset نمی‌شود تا از Data Leakage جلوگیری شود.


In [35]:

label_start_date = reference_date

label_end_date = (
    reference_date + pd.Timedelta(days=future_days)
)


print("Label Window Start:", label_start_date)

print("Label Window End:", label_end_date)

print(
    "Future Observation Period:",
    future_days,
    "days"
)

Label Window Start: 2011-09-10 00:00:00
Label Window End: 2011-12-09 00:00:00
Future Observation Period: 90 days


تعیین محدوده زمانی Label Window

In [36]:

feature_customer_ids = (
    df_feature_transactions["CustomerID"]
    .drop_duplicates()
)



label_window_mask = (
    df_valid_product_sales["InvoiceDate"].ge(label_start_date)
    &
    df_valid_product_sales["InvoiceDate"].lt(label_end_date)
)



cohort_mask = (
    df_valid_product_sales["CustomerID"].isin(
        feature_customer_ids
    )
)



df_label_transactions = df_valid_product_sales.loc[
    label_window_mask & cohort_mask
].copy()


print(
    "Feature Cohort Customers:",
    feature_customer_ids.nunique()
)

print(
    "Label Window Transactions:",
    len(df_label_transactions)
)

print(
    "Customers with Future Purchases:",
    df_label_transactions["CustomerID"].nunique()
)

print(
    "Unique Future Invoices:",
    df_label_transactions["InvoiceNo"].nunique()
)

Feature Cohort Customers: 2783
Label Window Transactions: 110577
Customers with Future Purchases: 1719
Unique Future Invoices: 4681


## استخراج تراکنش‌های ۹۰ روز آینده

یک نکته مهم داریم.

قرار نیست تمام مشتریانی که در آینده خرید کرده‌اند وارد پروژه شوند.

ما فقط می‌خواهیم رفتار آینده مشتریانی را بررسی کنیم که در بازه ۱۸۰ روز گذشته مشتری فعال بوده‌اند.

به این گروه، Customer Cohort می‌گوییم.

یعنی گروه مشتریانی که مبنای تحلیل ما هستند

In [37]:

print(
    "Earliest Future Transaction:",
    df_label_transactions["InvoiceDate"].min()
)

print(
    "Latest Future Transaction:",
    df_label_transactions["InvoiceDate"].max()
)


assert (
    df_label_transactions["InvoiceDate"] >= label_start_date
).all()

assert (
    df_label_transactions["InvoiceDate"] < label_end_date
).all()


assert (
    df_label_transactions["CustomerID"]
    .isin(feature_customer_ids)
    .all()
)


assert (
    df_prep["InvoiceDate"].max() >= label_end_date
)


print("\nLabel Window validation passed!")

Earliest Future Transaction: 2011-09-11 10:35:00
Latest Future Transaction: 2011-12-08 20:01:00

Label Window validation passed!


اعتبارسنجی Label Window

بعد از استخراج اطلاعات باید مطمئن شویم هیچ اطلاعاتی خارج از بازه زمانی وارد نشده است

In [38]:

feature_customers_count = (
    df_feature_transactions["CustomerID"].nunique()
)

future_active_customers = (
    df_label_transactions["CustomerID"].nunique()
)

future_inactive_candidates = (
    feature_customers_count - future_active_customers
)


future_activity_percentage = (
    future_active_customers / feature_customers_count
) * 100


print("Feature Customers:", feature_customers_count)

print("Future Active Customers:", future_active_customers)

print(
    "Customers Without Future Purchase:",
    future_inactive_candidates
)

print(
    "Future Activity Percentage:",
    round(future_activity_percentage, 2),
    "%"
)

Feature Customers: 2783
Future Active Customers: 1719
Customers Without Future Purchase: 1064
Future Activity Percentage: 61.77 %


بررسی آماری Label Window

حالا می‌خواهیم ببینیم چه تعداد از مشتریان گروه اولیه در آینده خرید داشته‌اند.

## 11. Label Window Extraction

در این مرحله اطلاعات تراکنش‌های معتبر مشتریان را در بازه ۹۰ روز پس از Reference Date استخراج می‌کنیم.

هدف این مرحله بررسی رفتار آینده مشتریان و فراهم‌کردن اطلاعات لازم برای ساخت برچسب ریزش (Churn Label) است.

با توجه به Reference Date انتخاب‌شده (2011-09-10)، بازه برچسب‌گذاری به شکل زیر تعریف می‌شود:

* Start Date: 2011-09-10
* End Date: 2011-12-09 (Exclusive)

برای جلوگیری از ورود مشتریان جدید به فرایند برچسب‌گذاری، فقط مشتریانی بررسی می‌شوند که در Feature Window سابقه خرید معتبر داشته‌اند.

دیتافریم `df_label_transactions` شامل تراکنش‌های معتبر این مشتریان در بازه آینده است.

در این مرحله تعداد مشتریانی که حداقل یک خرید معتبر در بازه آینده داشته‌اند، محاسبه می‌شود.

مشتریانی که در این بازه هیچ خرید معتبری نداشته‌اند، کاندیدای دریافت برچسب Churn برابر 1 خواهند بود.

در مقابل، مشتریانی که حداقل یک خرید معتبر داشته‌اند، برچسب Churn برابر 0 دریافت خواهند کرد.

ساخت نهایی این برچسب در مرحله بعد انجام می‌شود.

**نکته مهم:** اطلاعات Label Window فقط برای تعیین متغیر هدف استفاده می‌شوند و نباید در استخراج ویژگی‌های گذشته مشتریان وارد شوند.


In [39]:

df_feature_work = df_feature_transactions.copy()



df_feature_work["LineTotal"] = (
    df_feature_work["Quantity"]
    * df_feature_work["UnitPrice"]
)



display(
    df_feature_work[
        [
            "CustomerID",
            "InvoiceNo",
            "StockCode",
            "Quantity",
            "UnitPrice",
            "LineTotal"
        ]
    ].head(10)
)

,CustomerID,InvoiceNo,StockCode,Quantity,UnitPrice,LineTotal
117394,16954,546431,22236,3,12.75,38.25
117395,16954,546431,22212,12,2.10,25.20
117396,16954,546431,21974,24,1.45,34.80
117397,16954,546431,84987,24,1.45,34.80
117398,16954,546431,84378,24,1.25,30.00
117399,16954,546431,84380,24,1.25,30.00
117400,16954,546431,21212,24,0.55,13.20
117401,16954,546431,21213,24,0.55,13.20
117402,16954,546431,21977,48,0.55,26.40
117403,16954,546431,22951,24,0.55,13.20


ایجاد LineTotal

برای محاسبه Monetary ابتدا باید ارزش مالی هر ردیف تراکنش را محاسبه کنیم.
فرمول:

LineTotal=Quantity×UnitPrice

In [40]:

customer_features = (
    df_feature_work
    .groupby("CustomerID")
    .agg(

        Frequency=(
            "InvoiceNo",
            "nunique"
        ),

        Monetary=(
            "LineTotal",
            "sum"
        ),

        UniqueProducts=(
            "StockCode",
            "nunique"
        ),

        FirstPurchase=(
            "InvoiceDate",
            "min"
        ),

        LastPurchase=(
            "InvoiceDate",
            "max"
        )

    )
    .reset_index()
)



display(customer_features.head(10))

,CustomerID,Frequency,Monetary,UniqueProducts,FirstPurchase,LastPurchase
0,12347,3,1603.68,50,2011-04-07 10:43:00,2011-08-02 08:48:00
1,12348,1,327.00,4,2011-04-05 10:47:00,2011-04-05 10:47:00
2,12352,2,240.66,7,2011-03-17 16:00:00,2011-03-22 16:08:00
3,12353,1,89.00,4,2011-05-19 17:47:00,2011-05-19 17:47:00
4,12354,1,1079.40,58,2011-04-21 13:11:00,2011-04-21 13:11:00
5,12355,1,459.40,13,2011-05-09 13:49:00,2011-05-09 13:49:00
6,12356,1,481.46,21,2011-04-08 12:33:00,2011-04-08 12:33:00
7,12358,1,404.86,11,2011-07-12 10:04:00,2011-07-12 10:04:00
8,12359,1,1109.32,47,2011-06-03 12:26:00,2011-06-03 12:26:00
9,12360,2,1378.28,62,2011-05-23 09:43:00,2011-08-19 10:10:00


## ساخت ویژگی‌های مشتری
سلول دوم: گروه‌بندی مشتریان و استخراج ویژگی‌ها

حالا می‌خواهیم تمام تراکنش‌های مربوط به هر مشتری را در یک گروه قرار دهیم.

برای این کار از groupby() استفاده می‌کنیم.

In [41]:

customer_features["Recency"] = (
    reference_date.normalize()
    - customer_features["LastPurchase"].dt.normalize()
).dt.days



customer_features["TenureDays"] = (
    customer_features["LastPurchase"].dt.normalize()
    - customer_features["FirstPurchase"].dt.normalize()
).dt.days



feature_columns = [
    "CustomerID",
    "Recency",
    "Frequency",
    "Monetary",
    "UniqueProducts",
    "TenureDays"
]


display(
    customer_features[feature_columns].head(10)
)

,CustomerID,Recency,Frequency,Monetary,UniqueProducts,TenureDays
0,12347,39,3,1603.68,50,117
1,12348,158,1,327.00,4,0
2,12352,172,2,240.66,7,5
3,12353,114,1,89.00,4,0
4,12354,142,1,1079.40,58,0
5,12355,124,1,459.40,13,0
6,12356,155,1,481.46,21,0
7,12358,60,1,404.86,11,0
8,12359,99,1,1109.32,47,0
9,12360,22,2,1378.28,62,88


محاسبه Recency و TenureDays
سلول سوم: ایجاد ویژگی‌های زمانی

تا اینجا ویژگی‌های Frequency و Monetary و UniqueProducts را ساختیم.

اما هنوز دو ویژگی زمانی داریم.

۱. Recency چیست؟

Recency نشان می‌دهد چند روز از آخرین خرید مشتری تا تاریخ مرجع گذشته است

In [42]:

print("Number of Customers:", len(customer_features))

print(
    "Unique CustomerIDs:",
    customer_features["CustomerID"].nunique()
)



print("\nMissing Values:")

print(customer_features.isna().sum())



print("\nFeature Statistics:")

display(
    customer_features[
        [
            "Recency",
            "Frequency",
            "Monetary",
            "UniqueProducts",
            "TenureDays"
        ]
    ].describe().round(2)
)



assert customer_features["CustomerID"].is_unique

assert customer_features["Frequency"].gt(0).all()

assert customer_features["Monetary"].gt(0).all()

assert customer_features["UniqueProducts"].gt(0).all()

assert customer_features["Recency"].between(
    1, history_days
).all()

assert customer_features["TenureDays"].ge(0).all()

assert customer_features["FirstPurchase"].ge(
    feature_start_date
).all()

assert customer_features["LastPurchase"].lt(
    reference_date
).all()

print("\nAll customer feature validations passed!")

Number of Customers: 2783
Unique CustomerIDs: 2783

Missing Values:
CustomerID        0
Frequency         0
Monetary          0
UniqueProducts    0
FirstPurchase     0
LastPurchase      0
Recency           0
TenureDays        0
dtype: int64

Feature Statistics:


,Recency,Frequency,Monetary,UniqueProducts,TenureDays
count,2783.00,2783.00,2783.00,2783.00,2783.00
mean,66.26,2.87,1286.67,42.84,51.59
std,51.79,4.18,4513.86,56.22,59.27
min,1.00,1.00,2.90,1.00,0.00
25%,20.00,1.00,251.66,13.00,0.00
50%,54.00,2.00,500.44,27.00,21.00
75%,106.00,3.00,1110.24,54.00,105.00
max,180.00,84.00,120028.60,1086.00,179.00



All customer feature validations passed!


اعتبارسنجی ویژگی‌ها
سلول چهارم: بررسی صحت Feature Dataset

حالا باید بررسی کنیم که ویژگی‌هایی که ساختیم منطقی باشند

## 12. Feature Engineering

در این مرحله تراکنش‌های مشتریان را به ویژگی‌های رفتاری در سطح مشتری تبدیل می‌کنیم.

در دیتاست اولیه، هر ردیف نشان‌دهنده یک قلم کالا در یک فاکتور است؛ اما در مسئله پیش‌بینی ریزش، واحد اصلی تحلیل مشتری است.

بنابراین با استفاده از `groupby()` و `agg()` تراکنش‌های هر مشتری تجمیع شده و برای هر مشتری یک ردیف ایجاد می‌شود.

ویژگی‌های استخراج‌شده عبارت‌اند از:

* **Recency:** تعداد روزهای گذشته از آخرین خرید مشتری تا Reference Date.
* **Frequency:** تعداد فاکتورهای منحصربه‌فرد مشتری در بازه ۱۸۰ روزه.
* **Monetary:** مجموع ارزش مالی محصولات خریداری‌شده، بر اساس Quantity × UnitPrice.
* **UniqueProducts:** تعداد کدهای محصول منحصربه‌فرد خریداری‌شده توسط مشتری.
* **TenureDays:** فاصله زمانی بین اولین و آخرین خرید مشاهده‌شده مشتری در بازه ۱۸۰ روزه.

در محاسبه Frequency از تعداد فاکتورهای منحصربه‌فرد استفاده شده است؛ زیرا هر فاکتور می‌تواند شامل چندین ردیف کالایی باشد.

همچنین Recency با استفاده از Reference Date پروژه محاسبه شده است، نه آخرین تاریخ موجود در کل دیتاست؛ این موضوع برای جلوگیری از نشت اطلاعات آینده اهمیت دارد.

با توجه به نتایج تحلیل همبستگی در مرحله Data Understanding، ویژگی‌های ActiveDays و TotalUnits فعلاً در مجموعه اولیه ویژگی‌ها قرار نگرفته‌اند؛ زیرا همبستگی بسیار زیادی با Frequency و Monetary داشتند.

در پایان، یکتایی شناسه مشتریان، محدوده ویژگی‌های زمانی و مقادیر گمشده بررسی می‌شوند.

خروجی این مرحله یک جدول در سطح مشتری است که مبنای خوشه‌بندی و مدل‌سازی ریزش مشتریان خواهد بود.


In [43]:

future_purchase_customer_ids = (
    df_label_transactions["CustomerID"]
    .dropna()
    .unique()
)


print(
    "Customers with future purchases:",
    len(future_purchase_customer_ids)
)

Customers with future purchases: 1719


شناسایی مشتریانی که در آینده خرید داشته‌اند

ابتدا باید شناسه مشتریانی را استخراج کنیم که حداقل یک خرید معتبر در بازه ۹۰ روزه داشته‌اند

In [44]:

customer_labels = customer_features[
    ["CustomerID"]
].copy()



customer_labels["Churn"] = (
    ~customer_labels["CustomerID"].isin(
        future_purchase_customer_ids
    )
).astype("int8")



display(customer_labels.head(10))

,CustomerID,Churn
0,12347,0
1,12348,0
2,12352,0
3,12353,1
4,12354,1
5,12355,1
6,12356,0
7,12358,0
8,12359,0
9,12360,0


ایجاد دیتاست برچسب مشتریان

حالا می‌خواهیم برای تمام مشتریانی که در Feature Window حضور داشتند، برچسب ریزش بسازیم.

In [45]:

churn_distribution = (
    customer_labels["Churn"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)


churn_report = pd.DataFrame({
    "Customer_Count": churn_distribution,

    "Percentage": (
        churn_distribution
        / len(customer_labels)
    ) * 100
})


churn_report.index.name = "Churn"

display(churn_report.round(2))

,Customer_Count,Percentage
Churn,,
0,1719,61.77
1,1064,38.23


بررسی توزیع برچسب‌ها

حالا باید بررسی کنیم چند مشتری ریزش کرده و چند مشتری ریزش نکرده‌اند

In [46]:

assert customer_labels["CustomerID"].is_unique

assert customer_labels["Churn"].notna().all()

assert customer_labels["Churn"].isin([0, 1]).all()


assert customer_labels["CustomerID"].isin(
    feature_customer_ids
).all()


future_active_mask = customer_labels["CustomerID"].isin(
    future_purchase_customer_ids
)

assert (
    customer_labels.loc[future_active_mask, "Churn"] == 0
).all()

assert (
    customer_labels.loc[~future_active_mask, "Churn"] == 1
).all()


print("Total Customers:", len(customer_labels))

print("Churned Customers:", (customer_labels["Churn"] == 1).sum())

print("Non-Churned Customers:", (customer_labels["Churn"] == 0).sum())

print("\nAll Churn Label validations passed!")

Total Customers: 2783
Churned Customers: 1064
Non-Churned Customers: 1719

All Churn Label validations passed!


اعتبارسنجی برچسب‌ها

قبل از استفاده از این برچسب‌ها باید مطمئن شویم درست ساخته شده‌اند

## 13. Churn Label Creation

در این مرحله متغیر هدف (Target Variable) پروژه را ایجاد می‌کنیم.

از آنجایی که دیتاست اولیه دارای ستون مشخصی برای ریزش مشتریان نیست، برچسب ریزش را بر اساس رفتار خرید مشتریان در بازه ۹۰ روز پس از Reference Date تعریف می‌کنیم.

قانون برچسب‌گذاری به شکل زیر است:

* Churn = 0: مشتری حداقل یک خرید معتبر در بازه ۹۰ روز آینده داشته است.
* Churn = 1: مشتری در بازه ۹۰ روز آینده هیچ خرید معتبری نداشته است.

ابتدا شناسه مشتریانی که در Label Window حداقل یک خرید معتبر داشته‌اند استخراج می‌شود.

سپس با مقایسه این شناسه‌ها با مشتریان موجود در Feature Window، برچسب ریزش برای هر مشتری ایجاد می‌شود.

مشتریانی که در بازه ۱۸۰ روز گذشته خرید داشته‌اند اما در بازه ۹۰ روز آینده خرید معتبری ندارند، به‌عنوان مشتریان ریزش‌کرده شناخته می‌شوند.

این برچسب یک تعریف عملیاتی از ریزش است و لزوماً به معنای ترک دائمی مشتری نیست.

در پایان، تعداد و درصد مشتریان هر کلاس محاسبه شده و صحت برچسب‌های ایجادشده بررسی می‌شود.

این متغیر در ادامه به‌عنوان Target برای آموزش مدل‌های Classification استفاده خواهد شد.


In [47]:

feature_ids = set(customer_features["CustomerID"])

label_ids = set(customer_labels["CustomerID"])


print("Customers in Features:", len(feature_ids))

print("Customers in Labels:", len(label_ids))

print(
    "Same Customer IDs:",
    feature_ids == label_ids
)


assert feature_ids == label_ids

assert customer_features["CustomerID"].is_unique

assert customer_labels["CustomerID"].is_unique

print("\nCustomerID validation passed!")

Customers in Features: 2783
Customers in Labels: 2783
Same Customer IDs: True

CustomerID validation passed!


بررسی شناسه مشتریان

قبل از ترکیب دو دیتافریم باید مطمئن شویم شناسه مشتریان در هر دو دیتافریم یکسان است

In [48]:

df_final = customer_features.merge(
    customer_labels,
    on="CustomerID",
    how="left",
    validate="one_to_one",
    indicator=True
)


print("Merge Results:")

print(df_final["_merge"].value_counts())



df_final = df_final.drop(columns="_merge")


display(df_final.head(10))

Merge Results:
_merge
both          2783
left_only        0
right_only       0
Name: count, dtype: int64


,CustomerID,Frequency,Monetary,UniqueProducts,FirstPurchase,LastPurchase,Recency,TenureDays,Churn
0,12347,3,1603.68,50,2011-04-07 10:43:00,2011-08-02 08:48:00,39,117,0
1,12348,1,327.00,4,2011-04-05 10:47:00,2011-04-05 10:47:00,158,0,0
2,12352,2,240.66,7,2011-03-17 16:00:00,2011-03-22 16:08:00,172,5,0
3,12353,1,89.00,4,2011-05-19 17:47:00,2011-05-19 17:47:00,114,0,1
4,12354,1,1079.40,58,2011-04-21 13:11:00,2011-04-21 13:11:00,142,0,1
5,12355,1,459.40,13,2011-05-09 13:49:00,2011-05-09 13:49:00,124,0,1
6,12356,1,481.46,21,2011-04-08 12:33:00,2011-04-08 12:33:00,155,0,0
7,12358,1,404.86,11,2011-07-12 10:04:00,2011-07-12 10:04:00,60,0,0
8,12359,1,1109.32,47,2011-06-03 12:26:00,2011-06-03 12:26:00,99,0,0
9,12360,2,1378.28,62,2011-05-23 09:43:00,2011-08-19 10:10:00,22,88,0


ترکیب دیتافریم‌ها با Merge

حالا که از یکسان بودن شناسه‌ها مطمئن شدیم، آن‌ها را ترکیب می‌کنیم

In [49]:

print("Final Dataset Shape:", df_final.shape)

print(
    "Unique Customers:",
    df_final["CustomerID"].nunique()
)



print("\nMissing Values:")

print(df_final.isna().sum())



print(
    "\nDuplicate CustomerIDs:",
    df_final["CustomerID"].duplicated().sum()
)



print("\nChurn Distribution:")

display(
    df_final["Churn"].value_counts().sort_index()
)



print("\nFeature Statistics:")

display(
    df_final[
        [
            "Recency",
            "Frequency",
            "Monetary",
            "UniqueProducts",
            "TenureDays"
        ]
    ].describe().round(2)
)

Final Dataset Shape: (2783, 9)
Unique Customers: 2783

Missing Values:
CustomerID        0
Frequency         0
Monetary          0
UniqueProducts    0
FirstPurchase     0
LastPurchase      0
Recency           0
TenureDays        0
Churn             0
dtype: int64

Duplicate CustomerIDs: 0

Churn Distribution:


Churn
0    1719
1    1064
Name: count, dtype: int64


Feature Statistics:


,Recency,Frequency,Monetary,UniqueProducts,TenureDays
count,2783.00,2783.00,2783.00,2783.00,2783.00
mean,66.26,2.87,1286.67,42.84,51.59
std,51.79,4.18,4513.86,56.22,59.27
min,1.00,1.00,2.90,1.00,0.00
25%,20.00,1.00,251.66,13.00,0.00
50%,54.00,2.00,500.44,27.00,21.00
75%,106.00,3.00,1110.24,54.00,105.00
max,180.00,84.00,120028.60,1086.00,179.00


بررسی کیفیت دیتاست نهایی

حالا باید مطمئن شویم دیتاست نهایی از نظر ساختار و کیفیت آماده است.

In [50]:

assert df_final["CustomerID"].is_unique

assert df_final["Churn"].notna().all()

assert df_final["Churn"].isin([0, 1]).all()

assert df_final["Recency"].between(
    1, history_days
).all()

assert df_final["Frequency"].gt(0).all()

assert df_final["Monetary"].gt(0).all()

assert df_final["UniqueProducts"].gt(0).all()

assert df_final["TenureDays"].ge(0).all()


model_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "UniqueProducts",
    "TenureDays"
]

assert df_final[model_features].notna().all().all()


print("Final Dataset Validation Passed!")

print("\nTotal Customers:", len(df_final))

print("Total Features:", len(model_features))

print("Target Variable: Churn")

Final Dataset Validation Passed!

Total Customers: 2783
Total Features: 5
Target Variable: Churn


اعتبارسنجی نهایی

In [51]:

model_columns = [
    "CustomerID",
    "Recency",
    "Frequency",
    "Monetary",
    "UniqueProducts",
    "TenureDays",
    "Churn"
]


df_model = df_final[model_columns].copy()

display(df_model.head())

print("\nModel Dataset Shape:", df_model.shape)

print("\nModel Dataset Columns:")

print(df_model.columns.tolist())

,CustomerID,Recency,Frequency,Monetary,UniqueProducts,TenureDays,Churn
0,12347,39,3,1603.68,50,117,0
1,12348,158,1,327.00,4,0,0
2,12352,172,2,240.66,7,5,0
3,12353,114,1,89.00,4,0,1
4,12354,142,1,1079.40,58,0,1



Model Dataset Shape: (2783, 7)

Model Dataset Columns:
['CustomerID', 'Recency', 'Frequency', 'Monetary', 'UniqueProducts', 'TenureDays', 'Churn']


## ایجاد دیتاست مخصوص مدل‌سازی

یک نکته مهم وجود دارد.

در df_final دو ستون FirstPurchase و LastPurchase را نیز نگه داشته‌ایم.

این ستون‌ها برای بررسی و تحلیل مفید هستند، اما فعلاً نمی‌خواهیم به‌عنوان ورودی مستقیم مدل استفاده شوند.

همچنین CustomerID شناسه مشتری است و نباید به‌عنوان ویژگی رفتاری وارد مدل شود.

## 14. Final Dataset Preparation

در این مرحله ویژگی‌های استخراج‌شده از رفتار گذشته مشتریان با برچسب ریزش آن‌ها ترکیب می‌شوند.

دو دیتافریم اصلی پروژه عبارت‌اند از:

* `customer_features`: شامل ویژگی‌های رفتاری مشتریان در بازه ۱۸۰ روز گذشته.
* `customer_labels`: شامل برچسب ریزش مشتریان بر اساس رفتار آن‌ها در بازه ۹۰ روز آینده.

برای ترکیب این دو مجموعه داده از تابع `merge()` و ستون مشترک `CustomerID` استفاده می‌شود.

قبل از ترکیب، یکتایی و یکسان بودن شناسه مشتریان در هر دو دیتافریم بررسی می‌شود.

پس از ترکیب، دیتاست نهایی شامل ویژگی‌های زیر خواهد بود:

* Recency
* Frequency
* Monetary
* UniqueProducts
* TenureDays
* Churn

در این ساختار هر ردیف نماینده یک مشتری است و ستون Churn متغیر هدف پروژه محسوب می‌شود.

در ادامه، کیفیت دیتاست از نظر مقادیر گمشده، رکوردهای تکراری، اعتبار برچسب‌ها و محدوده ویژگی‌ها بررسی می‌شود.

همچنین یک نسخه مخصوص مدل‌سازی ایجاد می‌شود که در آن شناسه مشتری و متغیر هدف از ویژگی‌های ورودی مدل تفکیک شده‌اند.

در نتیجه، دیتاست نهایی آماده ورود به مراحل بعدی پروژه، شامل Clustering و Classification خواهد بود.


In [52]:
from pathlib import Path
import pandas as pd


output_dir = Path("prepared_data")
output_dir.mkdir(parents=True, exist_ok=True)



output_file = output_dir / "customer_churn_dataset.csv"



df_model.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print("Dataset successfully saved!")
print("File Path:", output_file.resolve())
print("Dataset Shape:", df_model.shape)

Dataset successfully saved!
File Path: C:\Users\VIRA\Desktop\D_online_shop\Data-Mining-Online-Shop\03_Data Prepeation\prepared_data\customer_churn_dataset.csv
Dataset Shape: (2783, 7)


In [53]:

df_check = pd.read_csv(output_file)



print("Original Shape:", df_model.shape)

print("Saved File Shape:", df_check.shape)

print("\nSaved Dataset Columns:")
print(df_check.columns.tolist())



assert df_check.shape == df_model.shape

assert df_check.columns.tolist() == df_model.columns.tolist()

assert df_check["CustomerID"].nunique() == (
    df_model["CustomerID"].nunique()
)

assert df_check["Churn"].isin([0, 1]).all()


print("\nFile verification successful!")

Original Shape: (2783, 7)
Saved File Shape: (2783, 7)

Saved Dataset Columns:
['CustomerID', 'Recency', 'Frequency', 'Monetary', 'UniqueProducts', 'TenureDays', 'Churn']

File verification successful!


## 15. Export Prepared Dataset

در آخرین مرحله از Data Preparation، دیتاست نهایی که شامل ویژگی‌های رفتاری مشتریان و برچسب ریزش آن‌هاست، در قالب یک فایل CSV ذخیره می‌شود.

برای این کار از تابع `to_csv()` در کتابخانه Pandas استفاده می‌کنیم.

فایل خروجی با نام زیر ذخیره می‌شود:

`customer_churn_dataset.csv`

این فایل شامل هفت ستون است:

* CustomerID
* Recency
* Frequency
* Monetary
* UniqueProducts
* TenureDays
* Churn

پس از ذخیره‌سازی، فایل مجدداً بارگذاری شده و تعداد ردیف‌ها، ستون‌ها، شناسه‌های مشتریان و اعتبار برچسب‌ها بررسی می‌شود.

هدف این مرحله ایجاد یک فایل مستقل و قابل استفاده مجدد برای مراحل مدل‌سازی است.

در نتیجه، دیتاست نهایی آماده ورود به الگوریتم‌های Clustering و Classification خواهد بود.


_________________________________________________________________________________________________________   

# Data Preparation – Final Summary

## 1. هدف Data Preparation

هدف این مرحله تبدیل داده‌های خام تراکنش‌های فروشگاه آنلاین به یک دیتاست ساختاریافته در سطح مشتری است.

از آنجایی که هدف پروژه شناسایی مشتریان در معرض ریزش است، داده‌ها باید به شکلی آماده شوند که بتوان از رفتار خرید گذشته مشتریان ویژگی استخراج کرد و بر اساس رفتار آینده آن‌ها، برچسب ریزش ساخت.

---

## 2. اقدامات انجام‌شده در Data Preparation

### 2.1. Creating Working Dataset

ایجاد یک نسخه مستقل از دیتاست اصلی با نام `df_prep` برای انجام عملیات آماده‌سازی، بدون تغییر در داده‌های اولیه.

### 2.2. Data Type Correction

اصلاح نوع داده ستون‌ها بر اساس ماهیت آن‌ها:

* تبدیل شناسه‌ها و ستون‌های متنی به String.
* تبدیل CustomerID به Nullable Integer.
* تبدیل InvoiceDate به Datetime.
* اطمینان از عددی بودن Quantity و UnitPrice.

### 2.3. Duplicate Records Removal

شناسایی و حذف رکوردهای کاملاً تکراری با استفاده از `drop_duplicates()` و حفظ اولین نسخه از هر رکورد.

### 2.4. Missing Values Handling

بررسی مقادیر گمشده و تصمیم‌گیری بر اساس کاربرد هر ستون:

* رکوردهای فاقد CustomerID برای تحلیل مشتری‌محور کنار گذاشته و جداگانه نگهداری شدند.
* مقادیر گمشده Description بدون جایگزینی حفظ شدند؛ زیرا StockCode همچنان برای شناسایی محصول در دسترس است.

### 2.5. Cancellation Handling

شناسایی فاکتورهای لغوشده با بررسی پیشوند C در InvoiceNo.

فاکتورهای لغوشده به گروه‌های دارای CustomerID و فاقد CustomerID تفکیک شدند تا اطلاعات مربوط به رفتار لغو مشتریان از بین نرود.

### 2.6. Negative Quantity and UnitPrice Handling

مقادیر منفی بر اساس ماهیت تجاری آن‌ها بررسی و تفکیک شدند:

* لغو و مرجوعی مشتریان.
* تعدیلات داخلی و انبار.
* تعدیلات مالی مانند Adjust bad debt.

تمام مقادیر منفی به‌صورت خودکار خطا در نظر گرفته نشدند.

### 2.7. Non-Product Transactions Identification

کدهای غیرکالایی مانند POSTAGE، DOT، CARRIAGE، Manual، Discount و سایر تعدیلات شناسایی شدند.

همچنین کارت‌های هدیه با الگوی gift_ به‌صورت جداگانه نگهداری شدند.

این تفکیک برای جلوگیری از محاسبه اشتباه تعداد محصولات و مبلغ خرید انجام شد.

### 2.8. Valid Purchase Transactions

تراکنش‌های معتبر محصولات با شرایط زیر استخراج شدند:

* CustomerID موجود باشد.
* فاکتور لغوشده نباشد.
* Quantity مثبت باشد.
* UnitPrice مثبت باشد.
* InvoiceDate معتبر باشد.
* کد کالا در فهرست کدهای غیرکالایی یا کارت‌های هدیه نباشد.

خروجی این مرحله در دیتافریم `df_valid_product_sales` نگهداری شد.

---

## 3. Reference Date and Time Windows

برای تفکیک اطلاعات گذشته و آینده و جلوگیری از Data Leakage، تاریخ مرجع پروژه تعیین شد.

**Reference Date: 2011-09-10**

دو پنجره زمانی تعریف شد:

| Window         | Time Range                      | Purpose                  |
| -------------- | ------------------------------- | ------------------------ |
| Feature Window | 2011-03-14 تا قبل از 2011-09-10 | استخراج ویژگی‌های رفتاری |
| Label Window   | 2011-09-10 تا قبل از 2011-12-09 | تعیین وضعیت ریزش         |

در Feature Window رفتار گذشته مشتریان بررسی شد و Label Window صرفاً برای تعیین متغیر هدف مورد استفاده قرار گرفت.

### ایجاد دیتاست‌های زمانی

* `df_feature_transactions`: تراکنش‌های معتبر در بازه ۱۸۰ روز گذشته.
* `df_label_transactions`: تراکنش‌های معتبر در بازه ۹۰ روز آینده، فقط برای مشتریان موجود در Feature Window.

---

## 4. Feature Engineering

در این مرحله تراکنش‌های مشتریان به ویژگی‌های رفتاری تبدیل شدند.

با استفاده از `groupby()` و `agg()` تمام تراکنش‌های هر مشتری تجمیع شدند تا هر مشتری فقط یک ردیف در دیتاست ویژگی‌ها داشته باشد.

### ویژگی‌های استخراج‌شده

| Feature        | Description                                | Meaning                       |
| -------------- | ------------------------------------------ | ----------------------------- |
| Recency        | فاصله آخرین خرید تا Reference Date         | تازگی فعالیت مشتری            |
| Frequency      | تعداد فاکتورهای منحصربه‌فرد                | میزان تکرار خرید              |
| Monetary       | مجموع Quantity × UnitPrice                 | ارزش مالی خرید محصولات        |
| UniqueProducts | تعداد StockCodeهای منحصربه‌فرد             | تنوع محصولات خریداری‌شده      |
| TenureDays     | فاصله اولین و آخرین خرید در پنجره ویژگی‌ها | گستره زمانی فعالیت مشاهده‌شده |

### توضیح ویژگی‌های RFM

**Recency:** هرچه مقدار آن کمتر باشد، مشتری اخیراً خرید کرده است.

**Frequency:** هرچه مقدار آن بیشتر باشد، مشتری دفعات بیشتری خرید کرده است.

**Monetary:** هرچه مقدار آن بیشتر باشد، ارزش مالی خرید محصولات مشتری بیشتر بوده است.

ویژگی‌های ActiveDays و TotalUnits در مجموعه اولیه ویژگی‌ها قرار نگرفتند؛ زیرا در تحلیل قبلی همبستگی زیادی با Frequency و Monetary داشتند.

خروجی این مرحله در دیتافریم `customer_features` ذخیره شد.

---

## 5. Churn Label Creation

از آنجایی که دیتاست اولیه دارای ستون Churn نبود، متغیر هدف بر اساس رفتار خرید مشتریان در ۹۰ روز آینده ایجاد شد.

قانون برچسب‌گذاری:

| Condition | Churn | Meaning |
|---|
